# 06. Aggregated PyOD - Multi-Level Analysis

Compare anomaly detection algorithms on **aggregated data** (Information Portraits):

**Algorithms:**
- **IForest** - Isolation Forest (global anomalies)
- **LOF** - Local Outlier Factor (local/contextual anomalies, O(n²))

**Three levels of analysis:**
1. **Buyers** (36K) — 12 features
2. **Suppliers** (360K) — 7 features
3. **Pairs** (916K) — 7 features

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve()))
from utils import setup_matplotlib, DATA_DIR, RESULTS_DIR, validate_dataframe, jaccard as jaccard_sim

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from itertools import combinations
from collections import Counter

from src.data_loader import load_tenders
from src.detectors.pyod_detector import AggregatedPyOD

setup_matplotlib()
print("Libraries loaded.")

## 1. Load Data

In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────
YEARS = [2022, 2023, 2024, 2025]

PARAMS = {
    # ML contamination: expected fraction of anomalies.
    # Applied identically to IForest and LOF to ensure a fair comparison.
    "contamination": 0.05,

    # Pair-level analysis: only include buyer-supplier pairs with enough
    # contracts to make anomaly detection meaningful
    "min_contracts": 3,

    # Consensus: flag an entity if flagged by at least this many algorithms
    "consensus_min": 2,
}

In [2]:
# Load full dataset
tenders = load_tenders(years=[2022, 2023, 2024, 2025])
buyers = load_buyers()
suppliers = load_suppliers()

print(f"Tenders: {len(tenders):,}")
print(f"Buyers: {len(buyers):,}")
print(f"Suppliers: {len(suppliers):,}")

Scanning 2022...
Scanning 2023...
Scanning 2024...
Scanning 2025...
Loaded 12,877,960 records
Loaded buyers: 35,995
Loaded suppliers: 358,376
Tenders: 12,877,960
Buyers: 35,995
Suppliers: 358,376


## 2. Compare Algorithms on Buyer Level

Running on 36K buyers instead of 13M tenders makes O(n²) algorithms like LOF feasible.

In [ ]:
# ── Data validation ────────────────────────────────────────────────────────
validate_dataframe(tenders, ["tender_id", "buyer_id", "supplier_id"], "tenders")

In [3]:
# Main algorithms for aggregated analysis
algorithms = ["iforest", "lof"]

results_dict = {}
comparison_data = []

for algo in algorithms:
    print()
    print("=" * 60)
    print(f"Algorithm: {algo.upper()}")
    print("=" * 60)
    
    start_time = datetime.now()
    
    detector = AggregatedPyOD(algorithm=algo, contamination=0.05)
    results = detector.detect_buyers(tenders)
    
    elapsed = (datetime.now() - start_time).total_seconds()
    
    anomalies = results[results['anomaly'] == 1]
    
    comparison_data.append({
        'algorithm': algo,
        'anomalies': len(anomalies),
        'anomaly_rate': len(anomalies) / len(results) * 100,
        'mean_score': results['score'].mean(),
        'max_score': results['score'].max(),
        'time_sec': elapsed,
    })
    
    results_dict[algo] = set(anomalies['buyer_id'].tolist())
    
    # Feature importance (IForest only)
    if algo == 'iforest':
        fi = detector.feature_importances("buyers")
        if fi:
            print("Feature importance:")
            for feat, imp in fi.items():
                print(f"  {feat}: {imp:.4f}")
    
    print(f"Time: {elapsed:.1f}s")
    print(f"Anomalies: {len(anomalies):,} ({len(anomalies)/len(results)*100:.2f}%)")


Algorithm: IFOREST
AggregatedPyOD (IFOREST): Detecting anomalous BUYERS...
  Computing buyer features from tenders...
  Features: ['single_bidder_rate', 'competitive_rate', 'avg_discount_pct', 'supplier_diversity_index', 'total_tenders', 'avg_value', 'total_value', 'cpv_concentration', 'avg_award_days', 'weekend_rate', 'value_variance_coeff', 'q4_rate']
  Buyers: 35,995
  Anomalies: 1,800 (5.0%)
Feature importance:
  total_value: 0.1213
  avg_value: 0.1088
  value_variance_coeff: 0.1085
  cpv_concentration: 0.0956
  supplier_diversity_index: 0.0942
  total_tenders: 0.0901
  q4_rate: 0.0883
  avg_discount_pct: 0.0773
  single_bidder_rate: 0.0745
  weekend_rate: 0.0740
  competitive_rate: 0.0673
  avg_award_days: 0.0000
Time: 7.8s
Anomalies: 1,800 (5.00%)

Algorithm: LOF
AggregatedPyOD (LOF): Detecting anomalous BUYERS...
  Computing buyer features from tenders...
  Features: ['single_bidder_rate', 'competitive_rate', 'avg_discount_pct', 'supplier_diversity_index', 'total_tenders', 'avg

In [4]:
# Comparison table
comparison_df = pd.DataFrame(comparison_data)
print()
print("=" * 60)
print("COMPARISON RESULTS (Buyer-Level)")
print("=" * 60)
display(comparison_df)


COMPARISON RESULTS (Buyer-Level)


,algorithm,anomalies,anomaly_rate,mean_score,max_score,time_sec
0,iforest,1800,5.000695,0.250407,1.0,7.842950
1,lof,1800,5.000695,0.159087,1.0,7.772125


In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Anomaly count
ax1 = axes[0]
comparison_df.plot(kind='bar', x='algorithm', y='anomalies', ax=ax1, color='coral', legend=False)
ax1.set_title('Anomalies Detected by Algorithm')
ax1.set_xlabel('')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)
ax1.axhline(y=comparison_df['anomalies'].mean(), color='red', linestyle='--', label='Mean')
ax1.legend()

# Mean score
ax2 = axes[1]
comparison_df.plot(kind='bar', x='algorithm', y='mean_score', ax=ax2, color='steelblue', legend=False)
ax2.set_title('Mean Anomaly Score by Algorithm')
ax2.set_xlabel('')
ax2.set_ylabel('Score')
ax2.tick_params(axis='x', rotation=45)

# Execution time
ax3 = axes[2]
comparison_df.plot(kind='bar', x='algorithm', y='time_sec', ax=ax3, color='green', legend=False)
ax3.set_title('Execution Time by Algorithm')
ax3.set_xlabel('')
ax3.set_ylabel('Seconds')
ax3.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "aggregated_pyod_buyer_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()

## 3. Algorithm Agreement Analysis

In [6]:
from itertools import combinations

print("Pairwise Agreement (Jaccard Similarity):")
print("=" * 60)

jaccard_matrix = pd.DataFrame(index=algorithms, columns=algorithms, dtype=float)

for algo1, algo2 in combinations(algorithms, 2):
    set1 = results_dict[algo1]
    set2 = results_dict[algo2]
    intersection = len(set1 & set2)
    union = len(set1 | set2)
    jaccard = intersection / union if union > 0 else 0
    
    jaccard_matrix.loc[algo1, algo2] = jaccard
    jaccard_matrix.loc[algo2, algo1] = jaccard
    
    print(f"{algo1:8} vs {algo2:8}: {jaccard:.3f} ({intersection:,} common)")

# Fill diagonal
for algo in algorithms:
    jaccard_matrix.loc[algo, algo] = 1.0

Pairwise Agreement (Jaccard Similarity):
iforest  vs lof     : 0.116 (375 common)


In [ ]:
# Heatmap of algorithm agreement
plt.figure(figsize=(10, 8))
sns.heatmap(jaccard_matrix.astype(float), annot=True, fmt='.2f', cmap='RdYlGn',
            square=True, linewidths=0.5, vmin=0, vmax=1)
plt.title('Algorithm Agreement (Jaccard Similarity)\nGreen = high agreement, Red = low')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / "aggregated_pyod_agreement_heatmap.png"), dpi=150, bbox_inches='tight')
plt.show()

## 4. Consensus Analysis

In [8]:
from collections import Counter

# Count how many algorithms flagged each buyer
all_flagged = []
for algo, buyer_ids in results_dict.items():
    all_flagged.extend(buyer_ids)

flag_counts = Counter(all_flagged)

# Consensus levels
consensus_levels = {}
for threshold in range(1, len(algorithms) + 1):
    count = len([b for b, c in flag_counts.items() if c >= threshold])
    consensus_levels[f">={threshold} algorithms"] = count

print("Consensus Analysis:")
print("=" * 40)
for level, count in consensus_levels.items():
    print(f"{level}: {count:,} buyers")

Consensus Analysis:
>=1 algorithms: 3,225 buyers
>=2 algorithms: 375 buyers


In [9]:
# High-consensus anomalies (flagged by both algorithms)
high_consensus_buyers = [b for b, c in flag_counts.items() if c >= 2]

print(f"High-consensus anomalies (both IForest + LOF): {len(high_consensus_buyers):,} buyers")

if high_consensus_buyers:
    # Merge with buyer details
    high_consensus_df = buyers[buyers['buyer_id'].isin(high_consensus_buyers)].copy()
    high_consensus_df['algorithms_flagged'] = high_consensus_df['buyer_id'].map(flag_counts)
    high_consensus_df = high_consensus_df.sort_values('total_value', ascending=False)
    
    print(f"\nTotal tender value: {high_consensus_df['total_value'].sum()/1e9:.2f} B UAH")
    print(f"Mean single_bidder_rate: {high_consensus_df['single_bidder_rate'].mean()*100:.1f}%")
    
    print("\nTop 10 by total value:")
    display(high_consensus_df.head(10)[['buyer_id', 'buyer_name', 'algorithms_flagged', 
                                         'total_tenders', 'total_value', 'single_bidder_rate']])

High-consensus anomalies (both IForest + LOF): 375 buyers

Total tender value: 573.67 B UAH
Mean single_bidder_rate: 18.8%

Top 10 by total value:


,buyer_id,buyer_name,algorithms_flagged,total_tenders,total_value,single_bidder_rate
11,20588716,"Приватне акціонерне товариство ""Укргідроенерго""",2,9054,1.213637e+11,0.241777
220,30019801,"Акціонерне товариство ""Укртрансгаз""",2,3488,4.823310e+10,0.258706
7344,00034022,Міністерство оборони України (Головне управлін...,2,509,4.250234e+10,0.000000
9602,25878206,СЛУЖБА ВІДНОВЛЕННЯ ТА РОЗВИТКУ ІНФРАСТРУКТУРИ ...,2,369,2.550868e+10,0.247059
1698,21467647,Департамент регіонального розвитку Київської о...,2,1581,1.823459e+10,0.106576
500,00130694,АТ Вінницяобленерго,2,2615,1.619532e+10,0.000000
155,03366144,ДЕПАРТАМЕНТ ЖИТЛОВО-КОМУНАЛЬНОГО ГОСПОДАРСТВА ...,2,3812,1.562750e+10,0.000000
14311,03359026,"Комунальна корпорація ""Київавтодор""",2,202,1.491431e+10,0.000000
10260,43493483,Департамент дорожнього господарства Львівської...,2,337,1.400047e+10,0.119403
1327,16286441,"Державне підприємство ""Поліграфічний комбінат ...",2,1781,1.351532e+10,0.109626


## 5. LOF vs Global Algorithms

LOF (Local Outlier Factor) detects **local anomalies** - entities that are unusual compared to their neighbors.
IForest and others detect **global anomalies** - entities unusual compared to the entire dataset.

In [10]:
# LOF-only anomalies (local but not global)
lof_set = results_dict['lof']
iforest_set = results_dict['iforest']

lof_only = lof_set - iforest_set
iforest_only = iforest_set - lof_set
both = lof_set & iforest_set

print("LOF vs IForest Comparison:")
print("=" * 40)
print(f"LOF only (local anomalies):     {len(lof_only):,}")
print(f"IForest only (global anomalies): {len(iforest_only):,}")
print(f"Both (local + global):           {len(both):,}")

LOF vs IForest Comparison:
LOF only (local anomalies):     1,425
IForest only (global anomalies): 1,425
Both (local + global):           375


In [11]:
# Analyze LOF-only anomalies
if lof_only:
    lof_only_df = buyers[buyers['buyer_id'].isin(lof_only)].copy()
    
    print("LOF-only anomalies (unusual in local context but not globally extreme):")
    print(f"  Count: {len(lof_only_df):,}")
    print(f"  Mean single_bidder_rate: {lof_only_df['single_bidder_rate'].mean()*100:.1f}%")
    print(f"  Mean total_tenders: {lof_only_df['total_tenders'].mean():.0f}")
    print(f"  Total value: {lof_only_df['total_value'].sum()/1e9:.2f} B UAH")

LOF-only anomalies (unusual in local context but not globally extreme):
  Count: 1,425
  Mean single_bidder_rate: 2.5%
  Mean total_tenders: 240
  Total value: 640.13 B UAH


## 7. Supplier-Level Analysis (360K)

In [12]:
# Supplier-level analysis
supplier_algorithms = ["iforest", "lof"]

supplier_results_dict = {}
supplier_comparison_data = []

for algo in supplier_algorithms:
    print()
    print("=" * 60)
    print(f"Algorithm: {algo.upper()} (Suppliers)")
    print("=" * 60)
    
    start_time = datetime.now()
    
    detector = AggregatedPyOD(algorithm=algo, contamination=0.05)
    results = detector.detect_suppliers(tenders)
    
    elapsed = (datetime.now() - start_time).total_seconds()
    
    anomalies = results[results['anomaly'] == 1]
    
    supplier_comparison_data.append({
        'algorithm': algo,
        'anomalies': len(anomalies),
        'anomaly_rate': len(anomalies) / len(results) * 100,
        'mean_score': results['score'].mean(),
        'time_sec': elapsed,
    })
    
    supplier_results_dict[algo] = set(anomalies['supplier_id'].tolist())
    
    print(f"Time: {elapsed:.1f}s")
    print(f"Anomalies: {len(anomalies):,} ({len(anomalies)/len(results)*100:.2f}%)")


Algorithm: IFOREST (Suppliers)
AggregatedPyOD (IFOREST): Detecting anomalous SUPPLIERS...
  Computing supplier features from tenders...
  Features: ['total_awards', 'total_value', 'avg_award_value', 'buyer_count', 'single_bidder_rate', 'avg_competitors', 'cpv_diversity']
  Suppliers: 358,377
  Anomalies: 17,919 (5.0%)
Time: 8.1s
Anomalies: 17,919 (5.00%)

Algorithm: LOF (Suppliers)
AggregatedPyOD (LOF): Detecting anomalous SUPPLIERS...
  Computing supplier features from tenders...
  Features: ['total_awards', 'total_value', 'avg_award_value', 'buyer_count', 'single_bidder_rate', 'avg_competitors', 'cpv_diversity']
  Suppliers: 358,377
  Anomalies: 17,919 (5.0%)
Time: 11.1s
Anomalies: 17,919 (5.00%)


In [13]:
# Supplier comparison table
supplier_comparison_df = pd.DataFrame(supplier_comparison_data)
print()
print("=" * 60)
print("COMPARISON RESULTS (Supplier-Level)")
print("=" * 60)
display(supplier_comparison_df)

# Supplier Jaccard
supplier_iforest = supplier_results_dict['iforest']
supplier_lof = supplier_results_dict['lof']
supplier_both = supplier_iforest & supplier_lof

print(f"\nIForest vs LOF (Suppliers):")
print(f"  IForest only: {len(supplier_iforest - supplier_lof):,}")
print(f"  LOF only: {len(supplier_lof - supplier_iforest):,}")
print(f"  Both: {len(supplier_both):,}")
jaccard = len(supplier_both) / len(supplier_iforest | supplier_lof) if len(supplier_iforest | supplier_lof) > 0 else 0
print(f"  Jaccard: {jaccard:.3f}")

# Supplier consensus
supplier_all_flagged = []
for algo, ids in supplier_results_dict.items():
    supplier_all_flagged.extend(ids)

supplier_flag_counts = Counter(supplier_all_flagged)

print()
print("Supplier Consensus:")
print("=" * 40)
for threshold in range(1, len(supplier_algorithms) + 1):
    count = len([s for s, c in supplier_flag_counts.items() if c >= threshold])
    print(f">={threshold} algorithms: {count:,} suppliers")


COMPARISON RESULTS (Supplier-Level)


,algorithm,anomalies,anomaly_rate,mean_score,time_sec
0,iforest,17919,5.000042,0.246834,8.141853
1,lof,17919,5.000042,0.019727,11.075835



IForest vs LOF (Suppliers):
  IForest only: 17,670
  LOF only: 17,670
  Both: 249
  Jaccard: 0.007

Supplier Consensus:
>=1 algorithms: 35,589 suppliers
>=2 algorithms: 249 suppliers


## 8. Pair-Level Analysis (Buyer-Supplier Relationships)

Detect anomalous buyer-supplier pairs with 3+ contracts.

In [14]:
# Pair-level analysis
pair_algorithms = ["iforest", "lof"]

pair_results_dict = {}
pair_comparison_data = []

for algo in pair_algorithms:
    print()
    print("=" * 60)
    print(f"Algorithm: {algo.upper()} (Pairs)")
    print("=" * 60)
    
    start_time = datetime.now()
    
    detector = AggregatedPyOD(algorithm=algo, contamination=0.05)
    results = detector.detect_pairs(tenders, min_contracts=3)
    
    elapsed = (datetime.now() - start_time).total_seconds()
    
    if len(results) > 0:
        anomalies = results[results['anomaly'] == 1]
        
        pair_comparison_data.append({
            'algorithm': algo,
            'total_pairs': len(results),
            'anomalies': len(anomalies),
            'anomaly_rate': len(anomalies) / len(results) * 100,
            'mean_score': results['score'].mean(),
            'time_sec': elapsed,
        })
        
        # Store as tuple set for consensus
        pair_results_dict[algo] = set(
            zip(anomalies['buyer_id'].tolist(), anomalies['supplier_id'].tolist())
        )
        
        print(f"Time: {elapsed:.1f}s")
        print(f"Anomalies: {len(anomalies):,} ({len(anomalies)/len(results)*100:.2f}%)")


Algorithm: IFOREST (Pairs)
AggregatedPyOD (IFOREST): Detecting anomalous PAIRS...
  Computing pair features from tenders...
  Pairs with 3+ contracts: 916,278
  Features: ['contracts_count', 'total_value', 'avg_value', 'single_bidder_rate', 'exclusivity_buyer', 'exclusivity_supplier', 'temporal_concentration']
  Anomalies: 45,814 (5.0%)
Time: 15.8s
Anomalies: 45,814 (5.00%)

Algorithm: LOF (Pairs)
AggregatedPyOD (LOF): Detecting anomalous PAIRS...
  Computing pair features from tenders...
  Pairs with 3+ contracts: 916,278
  Features: ['contracts_count', 'total_value', 'avg_value', 'single_bidder_rate', 'exclusivity_buyer', 'exclusivity_supplier', 'temporal_concentration']
  Anomalies: 45,814 (5.0%)
Time: 25.1s
Anomalies: 45,814 (5.00%)


In [15]:
# Pair comparison table
pair_comparison_df = pd.DataFrame(pair_comparison_data)
print()
print("=" * 60)
print("COMPARISON RESULTS (Pair-Level)")
print("=" * 60)
display(pair_comparison_df)

# Pair consensus
if pair_results_dict:
    pair_iforest = pair_results_dict['iforest']
    pair_lof = pair_results_dict['lof']
    pair_both = pair_iforest & pair_lof
    
    print(f"\nIForest vs LOF (Pairs):")
    print(f"  IForest only: {len(pair_iforest - pair_lof):,}")
    print(f"  LOF only: {len(pair_lof - pair_iforest):,}")
    print(f"  Both: {len(pair_both):,}")
    jaccard = len(pair_both) / len(pair_iforest | pair_lof) if len(pair_iforest | pair_lof) > 0 else 0
    print(f"  Jaccard: {jaccard:.3f}")
    
    pair_all_flagged = []
    for algo, pairs in pair_results_dict.items():
        pair_all_flagged.extend(pairs)
    
    pair_flag_counts = Counter(pair_all_flagged)
    
    print()
    print("Pair Consensus:")
    print("=" * 40)
    for threshold in range(1, len(pair_algorithms) + 1):
        count = len([p for p, c in pair_flag_counts.items() if c >= threshold])
        print(f">={threshold} algorithms: {count:,} pairs")


COMPARISON RESULTS (Pair-Level)


,algorithm,total_pairs,anomalies,anomaly_rate,mean_score,time_sec
0,iforest,916278,45814,5.000011,0.273765,15.811581
1,lof,916278,45814,5.000011,0.193113,25.077865



IForest vs LOF (Pairs):
  IForest only: 36,259
  LOF only: 36,259
  Both: 9,555
  Jaccard: 0.116

Pair Consensus:
>=1 algorithms: 82,073 pairs
>=2 algorithms: 9,555 pairs


## 9. Summary & Save Results

In [ ]:
# Summary of all levels
print("=" * 60)
print("MULTI-LEVEL ANALYSIS SUMMARY (IForest + LOF)")
print("=" * 60)

print("\n1. BUYER-LEVEL (36K, 12 features):")
print(f"   High-consensus (both IForest + LOF): {len([b for b, c in flag_counts.items() if c >= 2]):,} buyers")

print("\n2. SUPPLIER-LEVEL (360K, 7 features):")
print(f"   High-consensus (both): {len([s for s, c in supplier_flag_counts.items() if c >= 2]):,} suppliers")

if pair_results_dict:
    print("\n3. PAIR-LEVEL (916K, 7 features):")
    print(f"   High-consensus (both): {len([p for p, c in pair_flag_counts.items() if c >= 2]):,} pairs")

# Save all results
print()
print("Saving results...")

# Buyer results
comparison_df.to_csv(str(RESULTS_DIR / "aggregated_pyod_buyer_comparison.csv"), index=False)
high_consensus_df.to_csv(str(RESULTS_DIR / "aggregated_pyod_high_consensus_buyers.csv"), index=False)

# Supplier results
supplier_comparison_df.to_csv(str(RESULTS_DIR / "aggregated_pyod_supplier_comparison.csv"), index=False)
high_consensus_suppliers = [s for s, c in supplier_flag_counts.items() if c >= 2]
pd.DataFrame({'supplier_id': high_consensus_suppliers}).to_csv(
    str(RESULTS_DIR / "aggregated_pyod_high_consensus_suppliers.csv"), index=False
)

# Pair results
if pair_comparison_data:
    pair_comparison_df.to_csv(str(RESULTS_DIR / "aggregated_pyod_pair_comparison.csv"), index=False)
    high_consensus_pairs = [(b, s) for (b, s), c in pair_flag_counts.items() if c >= 2]
    pd.DataFrame(high_consensus_pairs, columns=['buyer_id', 'supplier_id']).to_csv(
        str(RESULTS_DIR / "aggregated_pyod_high_consensus_pairs.csv"), index=False
    )

print(f"Results saved to {RESULTS_DIR}")
print(f"\nCompleted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")